# ⚙️ Pertemuan 4 — Data Preparation for Machine Learning
### Mata Kuliah: Python Machine Learning

---

## Tujuan Pembelajaran

Setelah pertemuan ini, kamu bisa:
1. Menjelaskan mengapa model ML tidak bisa menerima data mentah
2. Melakukan encoding pada data kategorikal
3. Melakukan feature scaling (normalisasi dan standarisasi)
4. Membagi data menjadi training set dan test set dengan benar
5. Menyiapkan data siap pakai untuk model ML

---

## Recap Cepat — Quiz Kilat ⚡

Sebelum mulai, jawab pertanyaan berikut tanpa melihat catatan:

1. Apa fungsi `df.isnull().sum()`?
2. Kapan kita menggunakan median daripada mean untuk mengisi missing value?
3. Apa itu outlier? Bagaimana cara mendeteksinya?

---

In [ ]:
# ============================================================
# SETUP
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print("✅ Setup selesai!")

In [ ]:
# Load data bersih dari Pertemuan 3
try:
    df = pd.read_csv('titanic_cleaned.csv')
    print("✅ Data dari Pertemuan 3 berhasil dimuat!")
except FileNotFoundError:
    # Fallback: load dan bersihkan ulang
    df = sns.load_dataset('titanic')
    df = df.drop(columns=['deck'])
    df['age'] = df['age'].fillna(df['age'].median())
    df['embarked'] = df['embarked'].fillna(df['embarked'].mode()[0])
    df['embark_town'] = df['embark_town'].fillna(df['embark_town'].mode()[0])
    print("⚠️ File tidak ditemukan — data dimuat ulang dari seaborn.")

print(f"Shape: {df.shape}")
df.head()

---

## BAGIAN 1: Kenapa Model ML Tidak Bisa Langsung Menerima Data Kita?

Model ML pada dasarnya adalah **fungsi matematika**.  
Fungsi matematika hanya bisa memproses **angka**, bukan teks.

```
Contoh kolom 'sex': 'male', 'female', 'male' ...
Model tidak mengerti teks ini!
```

Selain itu, ada masalah skala:
```
Kolom 'age'  : 0 – 80  (skala kecil)
Kolom 'fare' : 0 – 512 (skala besar)

Model bisa 'terlalu fokus' pada kolom dengan skala besar!
```

Dua solusi utama:
1. **Encoding** — Mengubah teks/kategori menjadi angka
2. **Scaling** — Menyamakan skala semua fitur numerik

---

## BAGIAN 2: Pilih Fitur yang Relevan

Tidak semua kolom perlu digunakan. Kita pilih fitur yang **bermakna** untuk model.

In [ ]:
# Lihat semua kolom yang tersedia
print("Kolom yang tersedia:")
for col in df.columns:
    print(f"  - {col} ({df[col].dtype})")

In [ ]:
# Kolom yang akan DIBUANG (tidak berguna untuk prediksi)
kolom_buang = [
    'alive',       # Sama informasi dengan 'survived' (target kita)
    'who',         # Redundan dengan 'sex' dan 'age'
    'adult_male',  # Redundan dengan 'sex' dan 'age'
    'embark_town', # Sama informasi dengan 'embarked'
    'alone'        # Bisa dihitung dari kolom lain
]

df_prep = df.drop(columns=kolom_buang)
print(f"Kolom setelah seleksi: {df_prep.shape[1]} kolom")
print(df_prep.columns.tolist())

---

## BAGIAN 3: Encoding Variabel Kategorikal

Ada dua cara utama encoding:

| Metode | Kapan Digunakan | Contoh |
|--------|----------------|--------|
| **Label Encoding** | Kategori dengan urutan (ordinal) | 'kecil' → 0, 'sedang' → 1, 'besar' → 2 |
| **One-Hot Encoding** | Kategori tanpa urutan (nominal) | 'merah' → [1,0,0], 'biru' → [0,1,0] |

In [ ]:
# Lihat kolom kategorikal
print("Kolom kategorikal:")
for col in df_prep.select_dtypes(include=['object', 'category']).columns:
    print(f"  - '{col}': {df_prep[col].unique()}")

In [ ]:
# === LABEL ENCODING ===
# Kolom 'sex': hanya 2 nilai (binary) → Label Encoding cocok

le = LabelEncoder()
df_prep['sex_encoded'] = le.fit_transform(df_prep['sex'])

print("Mapping sex encoding:")
for cls, num in zip(le.classes_, le.transform(le.classes_)):
    print(f"  '{cls}' → {num}")

print("\nHasil:")
df_prep[['sex', 'sex_encoded']].head(6)

In [ ]:
# === ONE-HOT ENCODING ===
# Kolom 'embarked': 3 nilai (C, Q, S) tanpa urutan → One-Hot Encoding

embarked_dummies = pd.get_dummies(df_prep['embarked'], prefix='embarked', drop_first=True)
print("Kolom one-hot embarked:")
print(embarked_dummies.head(6))

In [ ]:
# Kolom 'class' (ordinal: First > Second > Third) → Label Encoding manual
class_mapping = {'First': 1, 'Second': 2, 'Third': 3}
df_prep['class_encoded'] = df_prep['class'].map(class_mapping)

print("Mapping class:")
df_prep[['class', 'class_encoded']].value_counts().reset_index()

In [ ]:
# Gabungkan hasil encoding dan hapus kolom aslinya
df_prep = pd.concat([df_prep, embarked_dummies], axis=1)
df_prep = df_prep.drop(columns=['sex', 'embarked', 'class', 'pclass'])

print(f"Kolom setelah encoding: {df_prep.shape[1]} kolom")
df_prep.head()

In [ ]:
# ✏️ LATIHAN Encoding:
# Cek apakah masih ada kolom bertipe object/category
# Lakukan encoding yang sesuai pada kolom tersebut

print("Kolom yang masih bertipe object/category:")
print(df_prep.select_dtypes(include=['object', 'category']).columns.tolist())

# Tulis kode encoding di bawah jika masih ada:


---

## BAGIAN 4: Feature Scaling

**Mengapa perlu scaling?**

Bayangkan kamu menilai kualitas makanan berdasarkan:
- Harga: Rp 10.000 – Rp 100.000  
- Rating: 1 – 5

Tanpa scaling, perbedaan harga (90.000) akan "mendominasi" rating (4).  
Model akan lebih terpengaruh harga daripada rating.

| Metode | Rumus | Range Hasil | Kapan Digunakan |
|--------|-------|-------------|----------------|
| **Min-Max Scaler** | (x - min) / (max - min) | 0 sampai 1 | Distribusi tidak normal, tidak ada outlier besar |
| **Standard Scaler** | (x - mean) / std | Sekitar 0 (rata-rata=0, std=1) | Distribusi normal / ada outlier |

In [ ]:
# Lihat distribusi fitur numerik sebelum scaling
fitur_numerik = ['age', 'fare', 'sibsp', 'parch']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, col in enumerate(fitur_numerik):
    df_prep[col].hist(ax=axes[0, i], color='#3498db', alpha=0.7, edgecolor='white', bins=20)
    axes[0, i].set_title(f'Sebelum: {col}', fontsize=11)
    axes[0, i].set_ylabel('Frekuensi')

plt.suptitle('Distribusi Sebelum & Sesudah Scaling', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Terapkan StandardScaler pada fitur numerik
scaler = StandardScaler()

# CATATAN: scaler.fit() hanya dipanggil pada training data!
# Di sini kita fit pada seluruh data dulu untuk demo
df_scaled = df_prep.copy()
df_scaled[fitur_numerik] = scaler.fit_transform(df_prep[fitur_numerik])

print("Statistik SEBELUM scaling:")
print(df_prep[fitur_numerik].describe().loc[['mean', 'std', 'min', 'max']])

print("\nStatistik SESUDAH scaling:")
print(df_scaled[fitur_numerik].describe().loc[['mean', 'std', 'min', 'max']].round(3))

In [ ]:
# Visualisasi perbedaan distribusi sebelum vs sesudah
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sebelum
df_prep[['age', 'fare']].plot(kind='box', ax=axes[0], 
                               title='Sebelum Scaling')
axes[0].set_ylabel('Nilai')

# Sesudah  
df_scaled[['age', 'fare']].plot(kind='box', ax=axes[1], 
                                 title='Sesudah StandardScaler')
axes[1].set_ylabel('Nilai (z-score)')

plt.tight_layout()
plt.show()

---

## BAGIAN 5: Train-Test Split

> **Aturan Emas ML:** Model tidak boleh "melihat" data test selama training.

Analogi: Kalau kamu belajar menggunakan soal ujian yang sama persis, nilai kamu tidak bisa dipercaya.

**Konsep Data Leakage:** Terjadi ketika informasi dari data test "bocor" ke proses training.  
Ini membuat performa model terlihat bagus di evaluasi, tapi buruk di dunia nyata.

```
Dataset Penuh (891 baris)
│
├── Training Set (80%) → Digunakan untuk melatih model
│       712 baris
│
└── Test Set (20%) → Digunakan untuk mengevaluasi model
        179 baris
```

In [ ]:
# Pisahkan fitur (X) dan target (y)
# Target = kolom yang ingin diprediksi

# Hapus kolom non-numerik yang tersisa sebelum modelling
df_final = df_scaled.select_dtypes(include=[np.number])

print("Kolom yang digunakan:")
print(df_final.columns.tolist())

In [ ]:
X = df_final.drop(columns=['survived'])  # Fitur
y = df_final['survived']                 # Target (yang ingin diprediksi)

print(f"Shape X (fitur): {X.shape}")
print(f"Shape y (target): {y.shape}")
print(f"\nDistribusi target:")
print(y.value_counts())
print(f"Tidak selamat: {(y==0).sum()} | Selamat: {(y==1).sum()}")

In [ ]:
# Train-Test Split
# random_state=42 → agar hasilnya reproducible (sama setiap kali dijalankan)
# stratify=y → pastikan proporsi kelas sama di train dan test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print("=" * 45)
print("HASIL TRAIN-TEST SPLIT")
print("=" * 45)
print(f"Training set: {X_train.shape[0]} baris ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:     {X_test.shape[0]} baris ({X_test.shape[0]/len(X)*100:.0f}%)")
print(f"\nDistribusi target di TRAIN:")
print(y_train.value_counts(normalize=True).round(3))
print(f"\nDistribusi target di TEST:")
print(y_test.value_counts(normalize=True).round(3))

---

## BAGIAN 6: Aturan Penting — Fit Scaler HANYA pada Training Data

> ⚠️ **Ini adalah kesalahan umum yang dilakukan banyak orang!**

Jika kamu fit scaler pada seluruh data (train + test), maka scaler mengetahui statistik dari test set.  
Ini adalah bentuk **data leakage**.

**Cara yang benar:**
```
1. Lakukan train_test_split DULU
2. scaler.fit() → HANYA pada X_train
3. scaler.transform() → pada X_train DAN X_test
```

In [ ]:
# Cara BENAR melakukan scaling setelah split

# Split dulu (tanpa scaling)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler HANYA pada training data
scaler_final = StandardScaler()
scaler_final.fit(X_train_raw)  # Belajar dari train saja!

# Transform keduanya menggunakan statistik dari train
X_train_scaled = scaler_final.transform(X_train_raw)
X_test_scaled = scaler_final.transform(X_test_raw)  # Gunakan stats dari train!

print("✅ Scaling dilakukan dengan benar!")
print(f"X_train shape: {X_train_scaled.shape}")
print(f"X_test shape:  {X_test_scaled.shape}")

---

## BAGIAN 7: Teaser — Model Pertama Kita! 🚀

Data kita sudah siap! Mari kita lihat sekilas hasilnya dengan model sederhana.  
Kita akan bahas detail model ini di pertemuan berikutnya.

In [ ]:
# Latih model Logistic Regression sederhana
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Prediksi pada test set
y_pred = model.predict(X_test_scaled)

# Evaluasi
akurasi = accuracy_score(y_test, y_pred)

print("=" * 45)
print("🎉 MODEL PERTAMA KITA!")
print("=" * 45)
print(f"Model: Logistic Regression")
print(f"Akurasi: {akurasi:.1%}")
print(f"\nDari {len(y_test)} penumpang test,")
print(f"model berhasil memprediksi {int(akurasi * len(y_test))} dengan benar!")
print("\nDi pertemuan berikutnya, kita akan memahami cara kerja model ini ☝️")

In [ ]:
# Visualisasi: Data Siap vs Hasil Prediksi
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Distribusi target aktual
pd.Series(y_test).value_counts().plot(kind='bar', ax=axes[0], 
    color=['#e74c3c', '#2ecc71'], alpha=0.8, edgecolor='white')
axes[0].set_title('Distribusi Aktual (y_test)', fontsize=12)
axes[0].set_xticklabels(['Tidak Selamat', 'Selamat'], rotation=0)
axes[0].set_ylabel('Jumlah')

# Distribusi hasil prediksi
pd.Series(y_pred).value_counts().plot(kind='bar', ax=axes[1], 
    color=['#e74c3c', '#2ecc71'], alpha=0.8, edgecolor='white')
axes[1].set_title('Distribusi Prediksi (y_pred)', fontsize=12)
axes[1].set_xticklabels(['Tidak Selamat', 'Selamat'], rotation=0)
axes[1].set_ylabel('Jumlah')

plt.suptitle(f'Akurasi Model: {akurasi:.1%}', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Simpan data siap ML untuk pertemuan berikutnya
pd.DataFrame(X_train_scaled, columns=X.columns).to_csv('X_train.csv', index=False)
pd.DataFrame(X_test_scaled, columns=X.columns).to_csv('X_test.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print("✅ Data siap ML disimpan!")
print("   - X_train.csv")
print("   - X_test.csv")
print("   - y_train.csv")
print("   - y_test.csv")

---

## ✏️ Latihan Mandiri

In [ ]:
# LATIHAN: Coba MinMaxScaler sebagai alternatif
# 1. Lakukan split 70-30 (bukan 80-20)
# 2. Fit MinMaxScaler pada X_train
# 3. Transform X_train dan X_test
# 4. Latih ulang model dan bandingkan akurasinya dengan StandardScaler

# Tulis kode di sini:


---

## 📝 Ringkasan Pertemuan 4 & Alur Lengkap

```
DATA MENTAH
    ↓
[P3] Cleaning: Handle Missing, Duplikat, Outlier, Inkonsistensi
    ↓
[P4] Seleksi Fitur: Buang kolom tidak relevan
    ↓
[P4] Encoding: Ubah kategori → angka
    ↓
[P4] Train-Test Split (80:20, stratify)
    ↓
[P4] Scaling: Fit pada TRAIN → Transform TRAIN & TEST
    ↓
DATA SIAP ML ✅
    ↓
[P5+] Model Training & Evaluation
```

---

**Pertemuan berikutnya: Memahami Algoritma ML dan melatih berbagai model!** 🤖